# Introduction to Generative AI

**You’ll learn**: what “tokens” are, how small text-gen models behave, how to shape outputs with prompts & parameters, and how to spot+reduce hallucinations & bias.

**Prereqs:**

- Basic Python (optional but helpful)
- A browser (ChatGPT/Claude UI or similar)
- (Optional) Google Colab account

**What to submit (one ZIP/PDF):**

1. A 1–2 page “Prompt Journal” (table or CSV) with your prompts, parameters, and results.
1. 3–5 screenshots from each lab’s checkpoints.
1. Short reflections requested in each lab (bullet points are fine).


## Lab 1 — Tokenization Explorer (15–25 mins)

**Goal**: See how text becomes tokens and why this matters for length, cost, and context windows.
**Tools**: Google Colab or local Python


**Setup (Colab or local CPU)**:


In [1]:
#!pip install -q transformers pandas
#!pip install --upgrade jupyter ipywidgets

### Steps

**1. Load a tokenizer**


In [2]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")  # quick, tiny tokenizer

**2. Try a few strings**


In [3]:
samples = [
    "Hello world!",
    "We’ll test emojis 😄 and numbers 123,456.",
    "۔ےہ رہﺷ ورتﺻﺑوﺧ کﯾا یچراک",
    "Δx = 0.01 in numerical methods",
]

for s in samples:
    ids = tok.encode(s, add_special_tokens=False)
    print(f"\nTEXT: {s}\nTOKENS: {len(ids)}\n{tok.convert_ids_to_tokens(ids)}")


TEXT: Hello world!
TOKENS: 3
['Hello', 'Ġworld', '!']

TEXT: We’ll test emojis 😄 and numbers 123,456.
TOKENS: 16
['We', 'âĢ', 'Ļ', 'll', 'Ġtest', 'Ġem', 'oj', 'is', 'ĠðŁĺ', 'Ħ', 'Ġand', 'Ġnumbers', 'Ġ123', ',', '456', '.']

TEXT: ۔ےہ رہﺷ ورتﺻﺑوﺧ کﯾا یچراک
TOKENS: 42
['Û', 'Ķ', 'Û', 'Ĵ', 'Û', 'ģ', 'ĠØ', '±', 'Û', 'ģ', 'ï', 'º', '·', 'ĠÙĪ', 'Ø±', 'Øª', 'ï', 'º', '»', 'ï', 'º', 'ĳ', 'ÙĪ', 'ï', 'º', '§', 'Ġ', 'Ú', '©', 'ï', '¯', '¾', 'Ø§', 'Ġ', 'Û', 'Į', 'Ú', 'Ĩ', 'Ø±', 'Ø§', 'Ú', '©']

TEXT: Δx = 0.01 in numerical methods
TOKENS: 10
['Î', 'Ķ', 'x', 'Ġ=', 'Ġ0', '.', '01', 'Ġin', 'Ġnumerical', 'Ġmethods']


**3. Measure token lengths for your own 5 prompts** (one should include code, one should include emojis, one should include non-English text)


In [4]:
import pandas as pd

rows = []
your_prompts = [
    "Summarize: Data pipelines vs ETL vs ELT.",
    "Translate to Urdu: Artificial intelligence enables…",
    "Write a polite email about a schedule change 🙂",
    "Generate 5 SQL SELECT examples on a ‘sales’ table.",
    "Turn these notes into bullet points: …",
]

for p in your_prompts:
    ids = tok.encode(p, add_special_tokens=False)
    rows.append({"prompt": p, "token_count": len(ids)})

df = pd.DataFrame(rows)
print(df.head())

                                              prompt  token_count
0           Summarize: Data pipelines vs ETL vs ELT.           13
1  Translate to Urdu: Artificial intelligence ena...           10
2     Write a polite email about a schedule change 🙂            9
3  Generate 5 SQL SELECT examples on a ‘sales’ ta...           16
4             Turn these notes into bullet points: …            8


**Checkpoint** (screenshot): table of your 5 prompts with token counts.<br>
**Reflection** (2–4 bullets): What kinds of text inflate token counts the most? Why does that matter?<br>
**Deliverable**: Add the table to your Prompt Journal (CSV or pasted into a doc).


## Lab 2 — Your First Local Text Generation (20–30 mins)

**Goal**: Generate text with a tiny local model and see how parameters change style/creativity.<br>
**Tools**: Colab or local Python (CPU). We’ll use a super-light model for speed.<br>
**Setup**:


In [5]:
#!pip install -q transformers accelerate torch --index-url https://download.pytorch.org/whl/cpu

In [6]:
from transformers import pipeline

# Inicializar el pipeline con gpt2 en lugar de distilgpt2
# gen = pipeline("text-generation", model="distilgpt2", device=-1)
gen = pipeline("text-generation", model="gpt2", device=-1)

Device set to use cpu


### Steps

**1. Baseline generation**


In [7]:
prompt = "In three sentences, explain why data quality is critical for effective analytics:"  # <- Change

# Generar texto con parámetros optimizados
out = gen(
    prompt,
    max_new_tokens=120,
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
    pad_token_id=50256,
    num_return_sequences=1,
)

# Imprimir la salida
print(out[0]["generated_text"])

In three sentences, explain why data quality is critical for effective analytics:

A. The data quality of individual users and data sets is important to provide meaningful insights into their behavior and to identify problems.

B. The data quality of a company and its employees is important to understand and correct problems.

C. Data quality is a tool used by organizations and companies to build their business and to develop customer experiences.

Data quality can help organizations and companies identify and fix problems and improve customer experience.

Data quality can also be used to address problems, such as:

Evaluating the effectiveness of data collection and analysis.




**2. Temperature & nucleus sampling**


In [8]:
def run(prompt, temp, top_p, top_k):
    out = gen(
        prompt,
        max_new_tokens=120,
        do_sample=True,
        temperature=temp,
        top_p=top_p,
        top_k=top_k,
        pad_token_id=50256,
        num_return_sequences=1,
    )
    return out[0]["generated_text"]


prompt = "Brainstorm 5 product ideas for a laptop sleeve brand that targets students:"  # <- change
for t, p, k in [(0.2, 1.0, 50), (0.7, 0.9, 50), (1.0, 0.9, 50)]:
    print("\n—", {"temperature": t, "top_p": p, "top_k": k})
    print(run(prompt, t, p, k))


— {'temperature': 0.2, 'top_p': 1.0, 'top_k': 50}
Brainstorm 5 product ideas for a laptop sleeve brand that targets students:

The first product idea is a laptop sleeve that is designed to fit a laptop sleeve. The sleeve is made of a soft, durable material that is easy to clean and dry. The sleeve is made of a flexible, durable material that is easy to clean and dry. The sleeve is made of a flexible, durable material that is easy to clean and dry. The sleeve is made of a flexible, durable material that is easy to clean and dry. The sleeve is made of a flexible, durable material that is easy to clean and dry. The sleeve is made of a flexible, durable material that

— {'temperature': 0.7, 'top_p': 0.9, 'top_k': 50}
Brainstorm 5 product ideas for a laptop sleeve brand that targets students:

A laptop sleeve brand that targets students:

A laptop sleeve brand that targets students:

A laptop sleeve brand that targets students:

A laptop sleeve brand that targets students:

A laptop sleeve

**3. Max tokens & repetition**


In [9]:
p2 = "Continue the list with new, non-repetitive ideas:"
print(run(p2, 0.8, 0.95, 50))

Continue the list with new, non-repetitive ideas:

Use the following to add a new line to the end of your program.

Example of line 1 to add a new line:

$ cat /path/to/file.pl /path/to/file.txt /path/to/file.txt | /path/to/file.pl

Note that the new line does not necessarily include a new filename.

Now you can use a different example of a new line:

$ cat /path/to/file.pl /path/to/file.txt /path/to/


**Checkpoint** (screenshots):

- Greedy vs sampled outputs for the same prompt
- A side-by-side of temperature 0.2 vs 1.0 results<br>

**Reflection** (3–6 bullets): When did outputs get creative vs off-topic? Which settings felt best?<br>
**Deliverable**: 3 prompt/parameter/result rows added to your Prompt Journal.


## Lab 3 — Prompt Patterns That Actually Work (30–40 mins)

**Goal**: Practice 5 foundational patterns that shape outputs—no secret tricks, just clear instructions.

**Tools**: Any chat LLM UI (ChatGPT/Claude, etc.) or your Lab 2 pipeline for short tasks.

**Patterns & Tasks** (copy-edit as needed):

1. **Role + Goal + Audience**

- **Prompt**: “You are a senior data engineer. Goal: explain ‘data lineage’ for non-technical managers in 150–180 words. Output: 3 short paragraphs with 1 example.”
- Acceptance check: Right audience? Right length? 1 concrete example?

2. **Clear Structure & Format**

- **Prompt**: "Convert the following notes into a JSON array of {topic, key_points, action}. Only valid JSON. Notes: "
- **Acceptance check**: Valid JSON? No extra text?

3. **Few Examples** (Few-shot)

- Provide 2 input→output pairs, then a new input:
- **Input**: "Raw: ‘pls fix meeting 2:30 tmrw w/ ops’ → Clean: ‘Please schedule a meeting with Operations tomorrow at 2:30 PM.’"
- Do that twice, then: "Raw: ‘move standup 9:15 to 9:45 plz’ → Clean:"
- **Acceptance check**: Mirrors style/format of examples?

4. **Constraints & Guardrails**

- **Prompt**: "Summarize the article in 70–90 words. Use bullet points. No speculative claims, no names unless present in the text. Text: "
- **Acceptance check**: Word window? Bullet points? No speculation?

5. **Output Schema for Reuse**

- **Prompt**: "Generate 5 test cases to validate a function that parses YYYY-MM-DD dates. Return a Markdown table with columns: case_id, input, expected, edge_case, rationale."
- **Acceptance check**: Table present, edge cases covered (leap year, invalid month, etc.)?

**Checkpoint** (screenshots): one successful output per pattern with the acceptance checklist ticked (you can annotate).

**Reflection** (3–5 bullets): Which pattern made the biggest quality difference? What did you change when the first try wasn’t good?

**Deliverable**: Add all 5 prompts & final outputs to your Prompt Journal.


In [10]:
# change
prompt_rga = "You are a senior data engineer. Goal: explain ‘data lineage’ for non-technical managers in 150–180 words. Output: 3 short paragraphs with 1 example."
print(run(prompt_rga, 0.8, 0.95, 50))

You are a senior data engineer. Goal: explain ‘data lineage’ for non-technical managers in 150–180 words. Output: 3 short paragraphs with 1 example. This is the work of an experienced data scientist (for example, a software engineer, a data scientist, a computer science researcher, a medical doctor, a data scientist, or a person who would be a data scientist in his own right).

I'd like to be able to show you this:

What you do here is that you write code and then you have this idea for something, and then you look at it. You don't know what you're doing.

You put it all together and go, "Okay, let's do this."

The results are


In [11]:
# add
prompt_csf = """
Convert the following notes into a JSON array of {topic, key_points, action}. Only valid JSON. Notes: 

"""
print(run(prompt_csf, 1.0, 0.9, 50))


Convert the following notes into a JSON array of {topic, key_points, action}. Only valid JSON. Notes: 


Note: To change the current name of the site, set the $site_name option to.


Example:

<http_admin_url= "https://www.facebook.com/groups/123426297914771601/posts/" > <id>user</id> </http_admin_url>

This project allows you to set and modify all the properties on a per user basis on the homepage for a single website.

Example:

<http_admin_url= "https://www.facebook.com/


In [12]:
prompt_fs = "pls fix meeting 2:30 tmrw w/ ops’ → Clean: ‘Please schedule a meeting with Operations tomorrow at 2:30 PM.’"
print(run(prompt_fs, 1.0, 0.9, 50))

pls fix meeting 2:30 tmrw w/ ops’ → Clean: ‘Please schedule a meeting with Operations tomorrow at 2:30 PM.’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’’


In [13]:
# add
prompt_cg = "Summarize the article in 70–90 words. Use bullet points. No speculative claims, no names unless present in the text. Text: "
print(run(prompt_cg, 1.0, 0.9, 50))

Summarize the article in 70–90 words. Use bullet points. No speculative claims, no names unless present in the text. Text: _______

Title: A Brief History of the World, Part I The Great Awakening, by Thomas Jefferson, 1776

Author: Jefferson, Thomas, and his sons Thomas, Thomas of Orange and Thomas of Jefferson

Publisher: University Press

Subtitle: A History of the World, Part II The Great Awakening

Author: Jefferson, Thomas Jefferson

Publisher: University Press

Subtitle: A History of the World, Part III The Great Awakening

Author: Jefferson, Thomas Jefferson

Publisher: University Press

Subtitle: A History of the World


In [14]:
prompt_osr = "Generate 5 test cases to validate a function that parses YYYY-MM-DD dates. Return a Markdown table with columns: case_id, input, expected, edge_case, rationale."
print(run(prompt_osr, 1.0, 0.9, 50))

Generate 5 test cases to validate a function that parses YYYY-MM-DD dates. Return a Markdown table with columns: case_id, input, expected, edge_case, rationale.

The resulting Markdown tables provide a good basis for learning about the syntax, design and semantics of the code, and the syntax and semantics of the algorithm. This article focuses mainly on the Markdown syntax and semantics.

Why We Do This

By using a lexer that parses YYYY-MM-DD dates and generates an original Markdown table, we can get an understanding of how to build a robust, consistent and flexible codebase that compiles with modern web browsers. If you have a tool like JIRA or the JIT that can help you with


## Lab 4 — Safety, Hallucinations & Bias Spot-Check (25–30 mins)

**Goal**: Learn safe habits from day 1. You’ll probe for errors ethically and practice safer prompting.

**Important**: Stay within platform policies. Do not try to bypass safety. Focus on noticing issues and applying safer prompts.

## Tasks

**1. Hallucination Probe**

- **Prompt**: "Who is Dr. Ayesha Qureshi, the Nobel laureate in Data Engineering (2019)? Provide sources."
- She doesn’t exist in that role—observe what the model does.
- Then fix: "If uncertain, say ‘I don’t know.’ Verify the claim first. If no reliable source exists, say that plainly."

**2. Citations Discipline**

- Ask for “3 sources on ‘vertical farming yield improvements 2018–2024’ with 1-line summary each.”
- **Acceptance check**: Do links look real? Are claims plausible? (You don’t need to browse—just sanity-check wording & domains.)
- **Safer re-prompt**: "Only include sources you’re highly confident are real, and label any uncertainty."

**3. PII Redaction**

- Provide a fake paragraph with names, phone numbers, and an address.
- Prompt: "Redact all PII. Replace with consistent placeholders like [NAME_1], [PHONE_1], [ADDRESS_1]. Return only the redacted text."

**4. Bias Scan** (Neutral Rewrite)

- **Prompt**: "Rewrite the following job ad to be neutral, inclusive, and skills-first. Remove gendered language and age proxies. Text: "
- **Acceptance check**: Did it remove biased phrasing and preserve requirements?

**Checkpoint** (screenshots):

- A hallucination example + your safer re-prompt result
- A redacted text example with placeholders

**Reflection** (5–7 bullets): What signals warned you something might be wrong? Which safety prompts helped most?

**Deliverable**: 2–3 before/after snippets added to your Prompt Journal.


In [15]:
# implement

## Lab 5 — Prompt Quality Scorer (15–20 mins)

**Goal**: Create a tiny heuristic “prompt rater” so you internalize what a good prompt includes.

**Steps**


In [16]:
def score_prompt(p: str) -> dict:
    criteria = {
        "goal": any(k in p.lower() for k in ["goal:", "task:", "you are", "act as"]),
        "context": any(
            k in p.lower() for k in ["context:", "given:", "assume", "background"]
        ),
        "constraints": any(
            k in p.lower() for k in ["limit", "no", "avoid", "exactly", "only", "must"]
        ),
        "format": any(
            k in p.lower()
            for k in ["json", "table", "markdown", "schema", "fields:", "columns"]
        ),
        "examples": any(
            k in p.lower()
            for k in ["example", "e.g.", "for instance", "input:", "output:"]
        ),
    }

    total = sum(criteria.values())
    return {"criteria": criteria, "score_5": total}


tests = [
    "Summarize this.",
    "You are a senior data engineer. Goal: explain data lineage to non-technical managers. Output: 3 bullets.",
    "Context: meeting notes below. Convert to JSON with fields {topic, decisions}. Only valid JSON. Examples: …",
]
for t in tests:
    print(t, "->", score_prompt(t))

Summarize this. -> {'criteria': {'goal': False, 'context': False, 'constraints': False, 'format': False, 'examples': False}, 'score_5': 0}
You are a senior data engineer. Goal: explain data lineage to non-technical managers. Output: 3 bullets. -> {'criteria': {'goal': True, 'context': False, 'constraints': True, 'format': False, 'examples': True}, 'score_5': 3}
Context: meeting notes below. Convert to JSON with fields {topic, decisions}. Only valid JSON. Examples: … -> {'criteria': {'goal': False, 'context': True, 'constraints': True, 'format': True, 'examples': True}, 'score_5': 4}


**Checkpoint**: Screenshot of your scores + one improved prompt that raises its score.

**Deliverable**: Paste your function & one improved prompt into your submission.


## Grading Rubric (10 pts total)

- **Prompt Journal completeness** (3 pts) — labs covered, parameters logged, reflections included.
- **Technical execution** (3 pts) — code runs / screenshots show outputs; JSON/table formats correct.
- **Prompt quality** (2 pts) — clear goals, constraints, formats; improved over iterations.
- **Safety awareness** (2 pts) — hallucination handling, PII redaction, bias-aware rewrite.


## Troubleshooting & Tips

- **Model too slow / memory errors?** Stick with distilgpt2 and short prompts (max_new_tokens<=80).
- **Weird or repetitive outputs?** Lower temperature or increase top_p discipline; reduce max_new_tokens.
- **Messy results?** Add explicit format instructions ("Only valid JSON", "Markdown table with columns …").
- **Hallucinations?** Ask the model to state uncertainty and avoid fabricating specifics.
